# BeamProject Training Notebook

This notebook triggers the multimodal training pipeline in Google Colab.
It is structured for a report-only workflow: train, evaluate, and save metrics/artifacts without checkpoints.

## 1. Set Up Colab Runtime and GPU

Verify that Colab is using an accelerated runtime before starting training.

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('CUDA device name:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU')

## 2. Install and Import Dependencies

Install project dependencies and import the core libraries used by the training notebook.

In [ ]:
import os
import sys
import json
from pathlib import Path

print('Notebook bootstrap imports ready')

## 3. Mount Google Drive and Configure Paths

Mount Drive, clone the repository if needed, and define the output paths for reports and artifacts.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

repo_url = 'https://github.com/ShaownBuetBme/BeamProject.git'
project_dir = Path('/content/BeamProject')
drive_output_dir = Path('/content/drive/MyDrive/beam_runs')
drive_output_dir.mkdir(parents=True, exist_ok=True)

if not project_dir.exists():
    !git clone {repo_url} {project_dir}
else:
    print('Repository already exists. Pulling latest changes...')
    !git -C {project_dir} pull --ff-only

%cd {project_dir}
!pip install -q -r {project_dir / 'requirements.txt'}

if str(project_dir / 'src') not in sys.path:
    sys.path.insert(0, str(project_dir / 'src'))

print('Project directory:', project_dir)
print('Drive output directory:', drive_output_dir)

## 4. Load and Validate Training Data

Check that the prepared NPZ dataset is available and inspect the fold structure before training.

In [ ]:
from beam.data.dataset import load_beam_npz

npz_path = project_dir / 'data_pipeline' / 'artifacts' / 'beam_multimodal.npz'
if not npz_path.exists():
    raise FileNotFoundError(f'NPZ not found at {npz_path}. Build it first inside the repo.')

data = load_beam_npz(npz_path)

print('NPZ path:', npz_path)
print('Image tensor shape:', data.x_img.shape)
print('Numeric tensor shape:', data.x_num.shape)
print('Target tensor shape:', data.y.shape)
print('Number of folds:', data.num_folds)
print('Numeric columns:', data.numeric_columns.tolist())
print('Target columns:', data.target_columns.tolist())

## 5. Define Model and Hyperparameters

Set the training configuration for the multimodal CNN+MLP baseline.

In [ ]:
run_mode = 'cv12'  # use 'single_fold' if you want a single held-out fold
config_path = project_dir / 'configs' / 'multimodal_baseline.yaml'
output_dir = drive_output_dir

print('Run mode:', run_mode)
print('Config:', config_path)
print('Output dir:', output_dir)

## 6. Build Training Loop with Checkpoints

The training loop is implemented in the repository CLI. This notebook triggers it directly, and the project is configured to run report-only (no checkpoints).

In [ ]:
!python {project_dir / 'train_multimodal_cli.py'} \
  --config {config_path} \
  --run-mode {run_mode} \
  --output-dir {output_dir}

## 7. Start Training and Stream Logs

The previous cell runs the experiment. Use Colab's output stream to watch the training progress and fold-level metrics.

## 8. Run Evaluation and Save Artifacts

After training completes, review the generated summary files and plots in the Drive output directory.

In [ ]:
run_dirs = sorted(output_dir.glob('*'))
print('Saved runs:')
for run_dir in run_dirs[-5:]:
    print(run_dir)

if run_dirs:
    latest_run = run_dirs[-1]
    print('\nLatest run directory:', latest_run)
    for rel_path in ['metrics_summary.json', 'fold_metrics.csv', 'all_predictions.csv']:
        candidate = latest_run / rel_path
        print(rel_path, 'exists:', candidate.exists())